In [29]:
%run init_client.py

In [30]:
# Helper functions
from tracemalloc import stop


def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "extra_body": {"temperature": temperature}
    }

    if system:
        params["system"] = system
    if stop:
        params["extra_body"]["stop"] = stop

    message = client.messages.create(**params)
    return message.content[0].text

In [31]:
# prompt = f"""
#     Please provide a solution to the following task.
#     {task}
# """

In [51]:
import json


def generate_dataset():
    prompt = """
        Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
        that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
        each representing task that requires Python, JSON, or a Regex to complete.

        Example output:
        ```json
        [
            {
                "task": "Description of task",
                "format": "python" or "json" or "regex",
                "solution_criteria": "Key criteria for evaluating the solution"
            },
            ...additional
        ]
        ```

        * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
        * Focus on tasks that do not require writing much code

        Please generate 3 objects.
    """

    messages = []
    add_assistant_message(messages, "```json")
    add_user_message(messages, prompt)
    answer = chat(messages, stop=["```"])
    return json.loads(answer.strip().removeprefix("```json").removesuffix("```").strip())


In [52]:
dataset = generate_dataset()
dataset

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)


In [ ]:
def run_prompt(test_case):
    """Merge the task from the test case into a prompt and then return the result."""
    prompt = f"""
        Please provide a solution to the following task.
        {test_case["task"]}

        * Respond only with Python, json, or regex.
        * Do not include any explanations or comments in your response.
    """

    messages = []
    add_assistant_message(messages, "```code")
    add_user_message(messages, prompt)
    answer = chat(messages, stop=["```"])
    return answer.strip().removeprefix("```").removesuffix("```").strip()

In [53]:
def grade_by_model(output, test_case):
    """Grade the output based on the test case."""
    eval_prompt = f"""
        You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

        Original Task:
        <task>
        {test_case["task"]}
        </task>

        Solution to Evaluate:
        <solution>
        {output}
        </solution>

        Criteria you should use to evaluate the solution:
        <criteria>
        {test_case["solution_criteria"]}
        </criteria>
        
        Output Format
        Provide your evaluation as a structured JSON object with the following fields, in this specific order:
        - "strengths": An array of 1-3 key strengths
        - "weaknesses": An array of 1-3 key areas for improvement
        - "reasoning": A concise explanation of your overall assessment
        - "score": A number between 1-10

        Respond with JSON. Keep your response concise and direct.
        Example response shape:
        {{
            "strengths": string[],
            "weaknesses": string[],
            "reasoning": string,
            "score": number
        }}
    """  
    messages = []
    add_assistant_message(messages, "```json")
    add_user_message(messages, eval_prompt)
    eval_result = chat(messages, stop=["```"])
    return json.loads(eval_result.strip().removeprefix("```json").removesuffix("```").strip())

In [45]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [49]:
def run_test_cases(test_case):
    """Call the run_prompt and then grade the result."""
    output = run_prompt(test_case)
    model_result = grade_by_model(output, test_case)
    syntax_score = grade_syntax(output, test_case)
    avg_score = (model_result["score"] + syntax_score) / 2
    return {
        "test_case": test_case,
        "output": output,
        "score": avg_score,
        "reasoning": model_result["reasoning"],
        "model_score": model_result["score"],
        "syntax_score": syntax_score
    }

In [40]:
from statistics import mean

def run_evals(dataset):
    """Loads the dataset and run_test_cases for each test case."""
    results = []
    for test_case in dataset:
        result = run_test_cases(test_case)
        results.append(result)

    average_score = mean(result["score"] for result in results);
    print(f"Average score: {average_score}")
    return results
    

In [54]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_evals(dataset)

print(json.dumps(results, indent=2))

Average score: 4.5
[
  {
    "test_case": {
      "task": "Write a Python function that takes an S3 bucket name and a prefix string, then returns a list of all object keys matching that prefix using boto3.",
      "format": "python",
      "solution_criteria": "Uses boto3 S3 client or resource, correctly applies the Prefix parameter in list_objects_v2, handles pagination or at least retrieves the first page of results, and returns a list of key strings."
    },
    "output": "python\nimport boto3\nfrom typing import Optional\n\n\ndef list_s3_objects_with_prefix(bucket_name: str, prefix: str, region_name: Optional[str] = None) -> list[str]:\n    \"\"\"\n    List all S3 object keys in a bucket that match a given prefix.\n\n    Args:\n        bucket_name: The name of the S3 bucket.\n        prefix: The prefix string to filter objects by.\n        region_name: Optional AWS region name (e.g., 'us-east-1').\n\n    Returns:\n        A list of object keys matching the prefix.\n\n    Raises:\n 